# 01. 탐색적 데이터 분석 & 모델링

`python -m pipeline.preprocess` 실행 후 사용하세요. 강의 1-3(NumPy), 1-4(Pandas), 4장(결정 트리·랜덤 포레스트·SHAP) 내용을 단계별로 보여주는 발표용 노트북입니다.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from pipeline.config import FEATURE_COLS, FEATURES, PROCESSED_DIR

df = pd.read_parquet(PROCESSED_DIR / 'dataset.parquet')
df.head()

## 1. 결측치와 기초 통계 (Pandas)

In [ ]:
print(df.shape)
print(df[FEATURE_COLS].isna().sum())
df[FEATURE_COLS].describe().T

In [ ]:
# 업종 대분류별 다음 분기 평균 폐업률
df.groupby('industry_group')['next_close_rate'].agg(['mean', 'median', 'count'])

## 2. 레이블 생성과 X, y Shape 검증 (NumPy)

In [ ]:
from pipeline.train import split_and_label, to_xy

train, test, thresholds = split_and_label(df)
X_train, y_train = to_xy(train)
X_test, y_test = to_xy(test)
print('업종별 임곗값:', thresholds)
print(f'X_train {X_train.shape} {X_train.dtype}, y_train {y_train.shape}, 양성 비율 {y_train.mean():.3f}')

## 3. 결정 트리 vs 랜덤 포레스트

`python -m pipeline.train` 결과(`artifacts/meta.json`)를 불러와 비교합니다.

In [ ]:
import json
from pipeline.config import ARTIFACT_DIR

meta = json.loads((ARTIFACT_DIR / 'meta.json').read_text())
pd.DataFrame({k: v['test'] for k, v in meta['results'].items()}).drop('confusion_matrix')

## 4. SHAP 해석

`python -m pipeline.explain` 으로 생성한 그림을 확인합니다.

In [ ]:
from IPython.display import Image, display
for name in ['shap_beeswarm', 'importance_vs_shap', 'waterfall_example']:
    display(Image(ARTIFACT_DIR / 'figures' / f'{name}.png'))